In [10]:
import os
import json
import time
import re
import warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
import google.generativeai as genai

GEMINI_API_KEY = "Enter API key"
genai.configure(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.6-flash"
model = genai.GenerativeModel(model_name=MODEL_NAME, generation_config={"temperature": 0.0})

INPUT_FILE = "Path of input file"
OUTPUT_FILE = "name of the output file"


def clean_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    name = re.sub(r'\s+', ' ', name.strip()).title()
    name = re.sub(r"\b([A-Za-z]+)'([a-z])", lambda m: m.group(1).title() + "'" + m.group(2).upper(), name)
    name = re.sub(r"\bOneill\b", "O'Neill", name, flags=re.IGNORECASE)
    name = re.sub(r"\bOconnor\b", "O'Connor", name, flags=re.IGNORECASE)
    name = re.sub(r"\bObrien\b", "O'Brien", name, flags=re.IGNORECASE)
    name = re.sub(r"\b([A-Z])\s+([A-Z][a-z]+)", r"\1. \2", name)
    return name


def clean_country(country: str) -> str:
    if not isinstance(country, str) or not country.strip():
        return ""
    c_clean = re.sub(r'\.', '', country.strip()).upper()
    lookup = {
        'USA': 'USA', 'US': 'USA', 'UNITED STATES': 'USA',
        'UK': 'United Kingdom', 'UNITED KINGDOM': 'United Kingdom',
        'IN': 'India', 'INDIA': 'India',
        'SWE': 'Sweden', 'SWEDEN': 'Sweden',
        'CAN': 'Canada', 'CANADA': 'Canada',
        'AUS': 'Australia', 'AUSTRALIA': 'Australia',
        'UAE': 'UAE', 'UNITED ARAB EMIRATES': 'UAE',
        'BRASIL': 'Brazil', 'BRAZIL': 'Brazil',
        'ITALIA': 'Italy', 'ITALY': 'Italy',
        'VIET NAM': 'Vietnam', 'VIETNAM': 'Vietnam',
        'CHINA': 'China', 'MEXICO': 'Mexico', 'GERMANY': 'Germany',
        'IRELAND': 'Ireland', 'RUSSIA': 'Russia', 'EGYPT': 'Egypt',
        'FRANCE': 'France', 'SPAIN': 'Spain', 'JAPAN': 'Japan',
        'SOUTH KOREA': 'South Korea', 'KOREA': 'South Korea',
        'CZECHIA': 'Czechia', 'KENYA': 'Kenya', 'TAIWAN': 'Taiwan',
        'GHANA': 'Ghana'
    }
    return lookup.get(c_clean, country.strip().title())


def clean_tabular_basics(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip() for c in df.columns]

    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

    for col in df.columns:
        col_lower = col.lower()
        series_str = df[col].astype(str).str.strip()

        # Email
        if "email" in col_lower or series_str.str.contains(r"^[\w\.-]+@[\w\.-]+\.\w+$").any():
            df[col] = df[col].str.lower()
            continue

        # Dates
        if any(d in col_lower for d in ["date", "time", "signup", "created"]):
            df[col] = pd.to_datetime(df[col], format="mixed", errors="coerce").dt.strftime("%Y-%m-%d")
            continue

        # Booleans
        bool_set = {'true', 'false', 'yes', 'no', 'y', 'n', '1', '0'}
        if series_str.str.lower().isin(bool_set).mean() > 0.6:
            df[col] = series_str.str.lower().map({
                'yes': True, 'y': True, 'true': True, '1': True,
                'no': False, 'n': False, 'false': False, '0': False
            }).fillna(False)
            continue

        # Prices / Currency -> Enforce strictly positive floats
        has_currency = series_str.str.contains(r'[$€£₹]').any()
        is_price_name = any(re.search(r'\b' + re.escape(w) + r'\b', col_lower) 
                            for w in ["price", "amount", "mrp", "cost", "total", "fee", "debit", "credit", "balance"])
        if has_currency or is_price_name:
            # Strip all minus signs, symbols, and commas
            cleaned_num = series_str.str.replace(r'[$€£₹,\s\-()]', '', regex=True).str.extract(r'(\d+(?:\.\d+)?)')[0]
            df[col] = pd.to_numeric(cleaned_num, errors="coerce").fillna(0.0).round(2)
            continue

        # Quantities
        is_count_name = any(re.search(r'\b' + re.escape(w) + r'\b', col_lower) 
                            for w in ["qty", "quantity", "stock", "count", "rating", "units", "score"])
        if is_count_name:
            cleaned_int = series_str.str.extract(r'(\d+)')[0]
            df[col] = pd.to_numeric(cleaned_int, errors="coerce").fillna(0).astype(int)
            continue

        # Names
        if any(k in col_lower for k in ["name", "holder", "customer", "client", "user", "person"]):
            df[col] = df[col].apply(clean_name)
            continue

        # Countries
        if any(k in col_lower for k in ["country", "nation", "region"]):
            df[col] = df[col].apply(clean_country)
            continue

    return df


def clean_json_response(raw_text: str):
    if not raw_text or not raw_text.strip():
        return {}
    cleaned = re.sub(r'^```(?:json)?', '', raw_text.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r'```$', '', cleaned.strip(), flags=re.MULTILINE).strip()
    match = re.search(r'([\{\[].*[\}\]])', cleaned, flags=re.DOTALL)
    if match:
        cleaned = match.group(1)
    try:
        return json.loads(cleaned)
    except Exception:
        return {}


def batch_decompose_composite(unique_items: list, is_finance: bool = True) -> tuple:
    target_keys = ["entity_or_merchant", "category", "channel"]
    prompt = f"""
Decompose each financial transaction into:
- 'entity_or_merchant': Normalized vendor/employer name (Title Case, remove store IDs/zip codes).
- 'category': Financial category (e.g. Payroll, Software/Cloud, Travel, Utilities, Supplies, Food/Drink, Rent).
- 'channel': Payment method (e.g. ACH, Wire, Card, Direct Deposit, Check, ATM).

Raw inputs:
{json.dumps(unique_items)}

Return strictly JSON mapping each input string to an object with keys {target_keys}:
{{
  "raw_string": {{"entity_or_merchant": "...", "category": "...", "channel": "..."}}
}}
"""
    decomp_map = {}
    try:
        res = model.generate_content(prompt)
        decomp_map = clean_json_response(res.text)
    except Exception as e:
        print(f"Gemini API note: {e}")

    fallback_map = {}
    for item in unique_items:
        if item in decomp_map:
            fallback_map[item] = decomp_map[item]
        else:
            tokens = item.split()
            fallback_map[item] = {
                "entity_or_merchant": tokens[0].title() if tokens else item.title(),
                "category": "General",
                "channel": "Direct"
            }
    return fallback_map, target_keys


def save_dataframe_safely(df: pd.DataFrame, output_path: str):
    path = Path(output_path)
    base_stem = path.stem
    parent = path.parent
    extension = path.suffix
    target_path = path
    counter = 1

    while True:
        try:
            df.to_csv(target_path, index=False)
            print(f"\nProcessing complete! Cleaned file saved to:\n{target_path}")
            break
        except PermissionError:
            target_path = parent / f"{base_stem}_{counter}{extension}"
            counter += 1


def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded dataset: {len(df)} rows.")

    # 1. Tabular clean (strips negatives, formats dates, names, countries)
    df = clean_tabular_basics(df)

    # 2. Decompose composite text
    composite_col = next((c for c in df.columns if any(k in c.lower() for k in ["desc", "detail", "item", "product", "memo"])), None)
    if composite_col:
        unique_composite = df[composite_col].dropna().unique().tolist()
        decomp_map, target_keys = batch_decompose_composite(unique_composite, is_finance=True)
        for key in target_keys:
            df[key] = df[composite_col].map(lambda x: decomp_map.get(x, {}).get(key, ""))

    save_dataframe_safely(df, OUTPUT_FILE)


if __name__ == "__main__":
    main()

Loaded dataset: 1000 rows.

Processing complete! Cleaned file saved to:
C:\Users\snehi\Downloads\cleaned_messy_healthcare.csv
